In [5]:
import pandas as pd
import os

# Scratch file for interactively visualizing *parallel-ml-bench* data

DATA_ROOT = 'data'
TEST_FILE = 'test_mpl_cores:260815-120506:home:ab3c6b6692273c761927291bb65dbe256fd5ee64:260815-120506.processed.jsonl'

def load_df(fname):
    df = pd.read_json(os.path.join(DATA_ROOT, fname), lines=True)
    df.set_index('bench')
    return df

df = load_df(TEST_FILE)
df


,tag,bench,args,config,cwd,exp,trials,procs,cmd,host,timestamp,elapsed,returncode,binary_bytes,binary_md5,warmup_result_secs,test_results_secs,compiler_md5
0,primes,primes,-N 100000000,mpl-tuple,mpl,time,1,1,/usr/bin/time -v bin/primes.mpl-tuple.bin @mpl...,home,2015-08-26 12:05:13.386900,19.230145,0,808104,03cc1f2f0ec7523181f5cf1b37cb59e1,"[0.6901, 0.6907, 0.6774, 0.6959000000000001, 0...","[0.6890000000000001, 0.6933, 0.6865, 0.6787000...",c0c52981b472f904937a6440b85168f2
1,primes,primes,-N 100000000,mpl-tuple,mpl,time,1,2,/usr/bin/time -v bin/primes.mpl-tuple.bin @mpl...,home,2015-08-26 12:05:32.619083,13.455066,0,808104,03cc1f2f0ec7523181f5cf1b37cb59e1,"[0.4118, 0.4107, 0.40850000000000003, 0.4027, ...","[0.41240000000000004, 0.4106, 0.4071, 0.4062, ...",c0c52981b472f904937a6440b85168f2
2,bfs,bfs,../inputs/rmat-10M-symm-bin --no-dir-opt,mpl-tuple,mpl,time,1,1,/usr/bin/time -v bin/bfs.mpl-tuple.bin @mpl pr...,home,2015-08-26 12:05:46.076302,124.257111,0,1041896,91791d899d39743105b366fa2bd176eb,[5.8143],"[5.9069, 5.7671, 5.8354, 5.7874, 5.7229, 5.827...",c0c52981b472f904937a6440b85168f2
3,bfs,bfs,../inputs/rmat-10M-symm-bin --no-dir-opt,mpl-tuple,mpl,time,1,2,/usr/bin/time -v bin/bfs.mpl-tuple.bin @mpl pr...,home,2015-08-26 12:07:50.335771,66.667405,0,1041896,91791d899d39743105b366fa2bd176eb,"[3.0139, 2.9754]","[2.9692, 2.9637000000000002, 2.9276, 3.0081, 2...",c0c52981b472f904937a6440b85168f2
4,primes,primes,-N 100000000,mpl-baseline,mpl,time,1,1,/usr/bin/time -v bin/primes.mpl-baseline.bin @...,home,2015-08-26 12:08:57.006106,19.688240,0,886144,f7949c864a7092b221485a3f3e3e0f85,"[0.6952, 0.7056, 0.6861, 0.6751, 0.7225, 0.746...","[0.7155, 0.6876, 0.721, 0.679, 0.6808000000000...",c0c52981b472f904937a6440b85168f2
5,primes,primes,-N 100000000,mpl-baseline,mpl,time,1,2,/usr/bin/time -v bin/primes.mpl-baseline.bin @...,home,2015-08-26 12:09:16.696627,13.488597,0,886144,f7949c864a7092b221485a3f3e3e0f85,"[0.41150000000000003, 0.418, 0.402400000000000...","[0.4076, 0.4072, 0.41440000000000005, 0.413500...",c0c52981b472f904937a6440b85168f2
6,bfs,bfs,../inputs/rmat-10M-symm-bin --no-dir-opt,mpl-baseline,mpl,time,1,1,/usr/bin/time -v bin/bfs.mpl-baseline.bin @mpl...,home,2015-08-26 12:09:30.187451,122.682824,0,1082440,b4857aa3b58ab1d5a62576bdc07089f9,[5.7449],"[5.8479, 5.6788, 5.7524, 5.7356, 5.6489, 5.758...",c0c52981b472f904937a6440b85168f2
7,bfs,bfs,../inputs/rmat-10M-symm-bin --no-dir-opt,mpl-baseline,mpl,time,1,2,/usr/bin/time -v bin/bfs.mpl-baseline.bin @mpl...,home,2015-08-26 12:11:32.873116,65.709570,0,1082440,b4857aa3b58ab1d5a62576bdc07089f9,"[2.9917, 2.9269]","[2.9259, 2.9352, 2.8936, 2.9539, 2.9116, 2.879...",c0c52981b472f904937a6440b85168f2


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from dataclasses import dataclass
from IPython.display import display

# Use system latex
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"]
})

CHARTS_ROOT = 'charts/'

CHECKSUM_FIELD = 'binary_md5'
COMPILER_NAME_FIELD = 'config'

def safe_std(all_abs_ratiios):
    # np.atleast_1d converts a scalar like 5 into an array [5]
    # If it is already a list, it converts it to a 1D array
    arr = np.atleast_1d(all_abs_ratiios)
    
    # Sample standard deviation is undefined for N <= 1, so return 0
    if len(arr) <= 1:
        return 0.0

    # ddof=1 calculates the sample standard deviation; omit it for population stddev
    return np.std(arr, ddof=1)
    

# Plots data from the parallel-ml-bench suite
def plot_parallel_bench(df, values='runTime', title='', out_filename='', abbrevs=('mlton-baseline', 'mlton')):
    # Remove columns we don't care about
    filtered = df[['bench', COMPILER_NAME_FIELD, values, CHECKSUM_FIELD]]
    # Remove benchmarks witn identical binaries
    filtered = df[filtered.groupby('bench')[CHECKSUM_FIELD].transform('nunique') > 1]
    # Drop older duplicate runs before pivoting
    # TODO: why does this happen?
    filtered = filtered.drop_duplicates(subset=['bench', COMPILER_NAME_FIELD], keep='last')

    pivot = filtered.pivot(index='bench', columns=COMPILER_NAME_FIELD, values=values)
    # Calculate the ratio (<1 is good, >1 is bad) between pairs of trials
    #
    # this is still correct if the 'trials' columns are scalar
    all_abs_ratios = pivot.apply(lambda row:
                              np.array(row[abbrevs[1]]) / np.array(row[abbrevs[0]]),
                              axis=1)
    mean_abs_ratios = all_abs_ratios.apply(np.mean)
    std_abs_ratios = all_abs_ratios.apply(safe_std)
    # Convert to relative_pct (<0% is good, >0% is bad)
    relative_pct = (mean_abs_ratios - 1) * 100
    ax = relative_pct.plot(kind='bar',
                           yerr=std_abs_ratios)

    # Calculate the absolute geomean (1.0 is neutral)
    abs_geomean = np.exp(np.mean(np.log(mean_abs_ratios.dropna())))
    # Scale to match the metric for relative_pct
    geomean_pct = (abs_geomean - 1) * 100
    # Add a text box
    textstr = f'Geomean: {geomean_pct:+.1f}\\%'
    props = dict(boxstyle='square,pad=0.5', facecolor='white', alpha=0.9, edgecolor='black', linewidth=0.5)
    ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='right', bbox=props)
    
    ax.set_title(title)
    ax.set_xlabel('Benchmark name')
    ax.set_ylabel(r'Relative \% $\frac{\mathrm{test}}{\mathrm{base}} - 1 \times 100\%$')
    ax.yaxis.set_major_formatter(ticker.PercentFormatter())
    path = os.path.join(CHARTS_ROOT, f'{out_filename}.pdf')
    print(f'NOT saving chart to {path}')
    #print(f'Saving chart to {path}')
    #plt.savefig(path, format='pdf', bbox_inches='tight')
    plt.show()



In [ ]:
@dataclass(frozen=True)
class PlotConfig:
    values_column: str
    title: str
    out_filename: str

FLAVOR_NAME = 'parallel_ml_bench'

def plot_mlton_vs_mlton(data, type_name):
    print(f'Plotting file for {type_name} flattening (MLton vs MLton): {data}')
    df = load_df(data)
    configs = [
        PlotConfig(values_column='test_results_secs',
                   title='Run time comparison', 
                   out_filename=f'{type_name}_{FLAVOR_NAME}_run_mlton_vs_mlton'),
      #  PlotConfig(values_column='compileTime',
      #             title='Compile time comparison',
      #             out_filename=f'{type_name}_mlton_compile_mlton_vs_mlton'),
      PlotConfig(values_column='binary_bytes',
                 title='Binary size comparison',
                 out_filename=f'{type_name}_mlton_size_mlton_vs_mlton'),
    ]
    for c in configs:
        plot_parallel_bench(df, values=c.values_column, title=c.title, out_filename=c.out_filename)


# Generate all parallel-ml-bench MLton-vs-MLton charts (for the two configs)
def plot_parallel_bench_tuple_mlton_vs_mlton():
    fname = 'cc_tuple_flatten_fixed_hash:260813-220330:flattening-tests:e957206262ad2a8b93398cdf777dd91275a74fbd:260813-220330.processed.jsonl'
    plot_mlton_vs_mlton(fname, 'tuple')

def plot_parallel_bench_conapp_mlton_vs_mlton():
    fname = 'cc_conapp_flatten_fixed_hash:260814-000801:flattening-tests:dfcc9e1798eddbe9a3d884b806fa2a946f27000d:260814-000801.processed.jsonl'
    plot_mlton_vs_mlton(fname, 'conapp')

In [ ]:
plot_parallel_bench_tuple_mlton_vs_mlton()
plot_parallel_bench_conapp_mlton_vs_mlton()